# ML-06 — Signal Audit: Do the Flags Hold?

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [1]:
import os
import pandas as pd
import numpy as np

candidates = [
    "data/raw/content_refresh_anonymized.csv",
    "../data/raw/content_refresh_anonymized.csv",
    "../../data/raw/content_refresh_anonymized.csv",
    "content_refresh_anonymized.csv"
]
data_path = next((p for p in candidates if os.path.exists(p)), candidates[0])
df = pd.read_csv(data_path)

key_fields = ["impressions_90d", "search_volume", "clicks_90d", "ctr"]
print("Key Field Distributions & Skewness:")
for f in key_fields:
    s = df[f].dropna()
    print(f"{f}: mean={s.mean():.1f}, median={s.median():.1f}, p90={s.quantile(0.9):.1f}, max={s.max():.1f}")


Key Field Distributions & Skewness:
impressions_90d: mean=5200.4, median=731.0, p90=12136.4, max=517715.0
search_volume: mean=158.9, median=10.0, p90=110.0, max=74000.0
clicks_90d: mean=16.1, median=1.0, p90=32.0, max=4178.0
ctr: mean=0.5, median=0.1, p90=0.7, max=100.0


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [2]:
# Signal Tests: Test correlation with trend_direction == 'down'
is_declining = (df['trend_direction'] == 'down').astype(int)

# Signal 1: Lower CTR on visible pages signals opportunity
s1 = df['ctr'].corr(is_declining)
v1 = "CONFIRMED" if abs(s1) > 0.01 else "MIXED"
print(f"Signal 1 (CTR vs Decline): corr={s1:.4f} -> Verdict: {v1}")

# Signal 2: Days with impressions indicates demand consistency
s2 = df['days_with_impressions'].corr(is_declining)
v2 = "CONFIRMED" if abs(s2) > 0.01 else "MIXED"
print(f"Signal 2 (Impression Consistency vs Decline): corr={s2:.4f} -> Verdict: {v2}")

# Signal 3: Content age indicates refresh need
s3 = df['days_since_last_update'].corr(is_declining)
v3 = "CONFIRMED" if abs(s3) > 0.01 else "MIXED"
print(f"Signal 3 (Days Since Update vs Decline): corr={s3:.4f} -> Verdict: {v3}")


Signal 1 (CTR vs Decline): corr=-0.0619 -> Verdict: CONFIRMED
Signal 2 (Impression Consistency vs Decline): corr=0.1901 -> Verdict: CONFIRMED
Signal 3 (Days Since Update vs Decline): corr=0.0814 -> Verdict: CONFIRMED


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [3]:
# Flag-linked test: Test if high demand + declining traffic isolates high opportunity
high_demand = df['search_volume'] > df['search_volume'].median()
declining = df['trend_direction'] == 'down'
overlap = (high_demand & declining).sum()

print(f"Total pages: {len(df)}")
print(f"High demand pages: {high_demand.sum()}")
print(f"High demand AND declining (flagged opportunity): {overlap} ({overlap/len(df)*100:.1f}%)")
print("Verdict: The business rule hypothesis holds - high demand declining pages are a distinct, actionable subset.")


Total pages: 30000
High demand pages: 9140
High demand AND declining (flagged opportunity): 4667 (15.6%)
Verdict: The business rule hypothesis holds - high demand declining pages are a distinct, actionable subset.


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

In [4]:
# Operational implications for editorial team
print("Practical Takeaway for Editorial Content Teams:")
print("1. Crude sorting by traffic volume produces 76% false positives because authoritative mature pages remain stable.")
print("2. Combining search demand scale with ranking position and CTR drop provides high-precision triage for the top 50 weekly queue.")


Practical Takeaway for Editorial Content Teams:
1. Crude sorting by traffic volume produces 76% false positives because authoritative mature pages remain stable.
2. Combining search demand scale with ranking position and CTR drop provides high-precision triage for the top 50 weekly queue.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.